## Generate variant-based annotation table

Builds one row per designed 80K MPRA variant (tested cCREs + gene-candidate sets +
controls) with:
- [x] NGN2 BCalm allelic activity (logFC, adj. p-value), significance, effect direction
- [x] variant position re-derived via ref/alt sequence alignment (not trusted as-is from
  design metadata)
- [x] Enformer prioritization class (high effect / low effect / random)
- [x] gnomAD v3.1.2 allele count/frequency + allele-frequency category (singleton / rare /
  common / very common)
- [x] transition/transversion substitution classification
- [x] TFBS disruption: FIMO/HOCOMOCO H13CORE PWM local + full bit scores for REF vs ALT,
  filtered to well-fitting motifs, one TF per variant (highest brain-expressed on ties)
- [x] element-level annotations inherited from the (already cleaned) element annotation
  table: gene_set, TSS distance, E2G (brain/cardiac), SCREEN cCRE, region-level eQTL,
  phastCons, singleton density
- [x] reference vs. alternate allele's *element* (region) activity + is_emVar flag
- [x] fine-mapped eQTL overlap at the exact variant position: GTEx (49 tissues), EMS
  (brain tissues), UKBB (94 traits, hg19)

In [1]:
import ast
import glob
import json
import math
import os
import re
import sys
import time
from collections import defaultdict
from os import listdir

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyranges as pr
import requests
import seaborn as sns
import yaml
from Bio import motifs
from Bio.Seq import Seq

sys.path.append("../00_helpful_functions")
import helpful_functions as hf

# column names
col_name = "name"
col_variant_class = "variant_class"
col_variant_pos = "variant_pos"
col_SPDI = "SPDI"
col_allele = "allele"

list_columns = [col_variant_class, col_variant_pos, col_SPDI, col_allele]
BASES = ["A", "C", "G", "T"]

config_path = "../cCRE_based_analysis/data/global80K_config.yaml"
with open(config_path) as conf:
    config = yaml.load(conf, Loader=yaml.FullLoader)

significant_threshold = 0.1
relative_bitscore_threshold = 0.75  # chosen over 0.8: many TFs sit just under 0.8 with no strong argument to exclude them
chosen_order = ["very common", "common", "rare", "singleton"]

writing = False  # set True to (re)write intermediate + final output tables
plotting = False  # set True to also show optional QC plots


def assert_condition(cond, msg):
    if not cond:
        raise AssertionError(f"\nSanity check failed:\n{msg}\n")


def normalize_name(name):
    """Sort ';'-joined header parts so BCalm/metadata/variant-map names line up regardless of allele order."""
    parts = sorted(part.strip() for part in name.split(";"))
    return ";".join(sorted(parts, reverse=True))

## Load BCalm variant results, variant map, and design metadata

In [2]:
input_directory = "data/"
NGN2_RESULTS_PATH = os.path.join(
    input_directory,
    "NGN2_variant_bcalm_80k_bbmap_251128_bcalm_normalized_variant_map_names_dev.tsv",
)
VARIANT_MAP_PATH = "data/2509_variant_map_region_based_controls.tsv.gz"
METADATA_PATH = "../cCRE_based_analysis/data/MPRA_251117_corrected_spdi.metadata.tsv.gz"

NGN2_df = pd.read_csv(NGN2_RESULTS_PATH, sep="\t")
variant_map_df = pd.read_csv(VARIANT_MAP_PATH, sep="\t")
variant_map_df["new_REF"] = variant_map_df["REF"].apply(normalize_name)
variant_map_df["new_ALT"] = variant_map_df["ALT"].apply(normalize_name)

metadata_df = pd.read_csv(METADATA_PATH, sep="\t")
for col in list_columns:
    metadata_df[col] = metadata_df[col].apply(hf.safe_eval)
metadata_df["normalized_name"] = metadata_df[col_name].apply(normalize_name)

print(f"NGN2 BCalm variant results: {NGN2_df.shape[0]} rows")
print(f"Variant map (base scaffold): {variant_map_df.shape[0]} rows")
print(f"Design metadata: {metadata_df.shape[0]} rows")

/tmp/ipykernel_738637/948183266.py:14: DtypeWarning: Columns (0: source, 1: info) have mixed types. Specify dtype option on import or set low_memory=False.
  metadata_df = pd.read_csv(METADATA_PATH, sep="\t")


NGN2 BCalm variant results: 40450 rows
Variant map (base scaffold): 48173 rows
Design metadata: 80215 rows


In [3]:
# --- Base scaffold: every variant from the variant map ends up in the final output ---
variant_effect_df = variant_map_df[["ID", "REF", "ALT", "new_REF", "new_ALT"]].copy()

metadata_exploded = (
    metadata_df.assign(normalized_name=metadata_df["normalized_name"].str.split(";"))
    .explode("normalized_name")
    .assign(normalized_name=lambda df: df["normalized_name"].str.strip())
)

metadata_ref = metadata_exploded[metadata_exploded["allele"].apply(hf.is_reference)][
    ["normalized_name", "sequence", "chr", "start", "end", "strand"]
].rename(columns={"normalized_name": "new_REF", "sequence": "ref_seq"})

metadata_alt = metadata_exploded[metadata_exploded["allele"].apply(hf.is_alternative)][
    ["normalized_name", "sequence", col_variant_pos, col_SPDI]
].rename(columns={"normalized_name": "new_ALT", "sequence": "alt_seq"})

variant_effect_df = variant_effect_df.merge(metadata_ref, on="new_REF", how="left")
variant_effect_df = variant_effect_df.merge(metadata_alt, on="new_ALT", how="left")
print(f"After merging REF/ALT metadata: {variant_effect_df.shape[0]} rows")

After merging REF/ALT metadata: 48173 rows


In [4]:
variant_effect_df

,ID,REF,ALT,new_REF,new_ALT,ref_seq,chr,start,end,strand,alt_seq,variant_pos,SPDI
0,cardiac_neuro_cava_random:ALT_SKI|ENSG00000157...,cardiac_neuro_cava_random:REF_SKI|ENSG00000157...,cardiac_neuro_cava_random:ALT_SKI|ENSG00000157...,cardiac_neuro_cava_random:REF_SKI|ENSG00000157...,cardiac_neuro_cava_random:ALT_SKI|ENSG00000157...,GTCCCAGCTCCCCACTGATGTGAAAGGTGGTGGTGAGTTAACAGCT...,chr1,2179507.0,2179777.0,+,GTCCCAGCTCCCCACTGATGTGAAAGGTGGTGGTGAGTTAACAGCT...,[83],[NC_000001.11:2179590:T:C]
1,cardiac_neuro_cava_random:ALT_SKI|ENSG00000157...,cardiac_neuro_cava_random:REF_SKI|ENSG00000157...,cardiac_neuro_cava_random:ALT_SKI|ENSG00000157...,cardiac_neuro_cava_random:REF_SKI|ENSG00000157...,cardiac_neuro_cava_random:ALT_SKI|ENSG00000157...,CCTGATCTGCCCTGTCCGTGACGCTTCTGCTCAGTAGCTGAGCACG...,chr1,2191262.0,2191532.0,+,CCTGATCTGCCCTGTCCGTGACGCTTCTGCTCAGTAGCTGAGCACG...,[181],[NC_000001.11:2191443:G:A]
2,cardiac_neuro_cava_random:ALT_SKI|ENSG00000157...,cardiac_neuro_cava_random:REF_SKI|ENSG00000157...,cardiac_neuro_cava_random:ALT_SKI|ENSG00000157...,cardiac_neuro_cava_random:REF_SKI|ENSG00000157...,cardiac_neuro_cava_random:ALT_SKI|ENSG00000157...,CCTCTGGGTGACCCGGAGAACACCAAGGCTGTGAGAAATGGGAGGC...,chr1,2191971.0,2192241.0,+,CCTCTGGGTGACCCGGAGAACACCAAGGCTGTGAGAAATGGGATGC...,[43],[NC_000001.11:2192014:G:T]
3,cardiac_neuro_cava_random:ALT_SKI|ENSG00000157...,cardiac_neuro_cava_random:REF_SKI|ENSG00000157...,cardiac_neuro_cava_random:ALT_SKI|ENSG00000157...,cardiac_neuro_cava_random:REF_SKI|ENSG00000157...,cardiac_neuro_cava_random:ALT_SKI|ENSG00000157...,CCATGCGGTGGCCACAGCCTCGGGTGAGTTCCGGTTCCAAAGTACC...,chr1,2192249.0,2192519.0,+,CCATGCGGTGGCCACAGCCTCGGGTGAGTTCCGGTTCCAAAGTACC...,[116],[NC_000001.11:2192365:T:G]
4,cardiac_neuro_cava_random:ALT_SKI|ENSG00000157...,cardiac_neuro_cava_random:REF_SKI|ENSG00000157...,cardiac_neuro_cava_random:ALT_SKI|ENSG00000157...,cardiac_neuro_cava_random:REF_SKI|ENSG00000157...,cardiac_neuro_cava_random:ALT_SKI|ENSG00000157...,GGACTCCGGTGCCTTCGCATTCCCGAGCTGTTTTTGCTTCTGGAAG...,chr1,2192936.0,2193206.0,+,GGACTCCGGTGCCTTCGCATTCCCGAGCTGTTTTTGCTTCTGGAAG...,[205],[NC_000001.11:2193141:G:A]
...,...,...,...,...,...,...,...,...,...,...,...,...,...
48168,GC_Mendelian_variants:ALT_chr7:156791255G>C|SH...,GC_Mendelian_variants:REF_chr7:156791255G>C|SHH,GC_Mendelian_variants:ALT_chr7:156791255G>C|SH...,GC_Mendelian_variants:REF_chr7:156791255G>C|SHH,GC_Mendelian_variants:ALT_chr7:156791255G>C|SH...,GAGATATGGCTTCATTTTCTGTAATAAACACTAAGATCAAAACATG...,chr7,156791119.0,156791389.0,+,GAGATATGGCTTCATTTTCTGTAATAAACACTAAGATCAAAACATG...,[154],[NC_000007.14:156791273:T:TTAAGGAAGTGATT]
48169,GC_Mendelian_variants:ALT_chr7:156791257G>A|SH...,GC_Mendelian_variants:REF_chr7:156791257G>A|SHH,GC_Mendelian_variants:ALT_chr7:156791257G>A|SH...,GC_Mendelian_variants:REF_chr7:156791257G>A|SHH,GC_Mendelian_variants:ALT_chr7:156791257G>A|SH...,GATATGGCTTCATTTTCTGTAATAAACACTAAGATCAAAACATGAC...,chr7,156791121.0,156791391.0,+,GATATGGCTTCATTTTCTGTAATAAACACTAAGATCAAAACATGAC...,[152],[NC_000007.14:156791273:T:TTAAGGAAGTGATT]
48170,GC_Mendelian_variants:ALT_chr7:156791274T>TTAA...,GC_Mendelian_variants:REF_chr7:156791274T>TTAA...,GC_Mendelian_variants:ALT_chr7:156791274T>TTAA...,GC_Mendelian_variants:REF_chr7:156791274T>TTAA...,GC_Mendelian_variants:ALT_chr7:156791274T>TTAA...,TGTAATAAACACTAAGATCAAAACATGACCCAAGTTAAATTTCCTT...,chr7,156791138.0,156791408.0,+,TGTAATAAACACTAAGATCAAAACATGACCCAAGTTAAATTTCCTT...,[135],[NC_000007.14:156791273:T:TTAAGGAAGTGATT]
48171,GC_Mendelian_variants:ALT_chr8:11703860G>T|GAT...,GC_Mendelian_variants:REF_chr8:11703860G>T|GATA4,GC_Mendelian_variants:ALT_chr8:11703860G>T|GAT...,GC_Mendelian_variants:REF_chr8:11703860G>T|GATA4,GC_Mendelian_variants:ALT_chr8:11703860G>T|GAT...,CGGGGCTGGGAGGATCCCCACTACCCCTGCCCAGGAACTAGCATCC...,chr8,11703724.0,11703994.0,+,CGGGGCTGGGAGGATCCCCACTACCCCTGCCCAGGAACTAGCATCC...,[166],[NC_000008.11:11703890:GGGGGGG:GGGGGG]


In [5]:
# --- Merge NGN2 BCalm results, using canonical column names from the start ---
NGN2_processed = NGN2_df[["variant_id", "adj.P.Val", "logFC"]].rename(
    columns={"adj.P.Val": "NGN2_variant_adj_p-value", "logFC": "NGN2_variant_logFC"}
)
NGN2_processed["NGN2_variant_has_readout"] = True

variant_effect_df = variant_effect_df.merge(
    NGN2_processed, left_on="ID", right_on="variant_id", how="left"
).drop(columns=["variant_id"])
variant_effect_df["NGN2_variant_has_readout"] = variant_effect_df[
    "NGN2_variant_has_readout"
].fillna(False)


def add_effect_direction(row):
    adj_p = row["NGN2_variant_adj_p-value"]
    logfc = row["NGN2_variant_logFC"]
    if pd.isna(adj_p) or adj_p >= significant_threshold:
        return "not_significant"
    if logfc > 0:
        return "upregulating"
    if logfc < 0:
        return "downregulating"
    return "no_change"  # exact logFC == 0 while significant


variant_effect_df["NGN2_variant_is_significant"] = (
    variant_effect_df["NGN2_variant_adj_p-value"] <= significant_threshold
)
variant_effect_df["NGN2_variant_effect_direction"] = variant_effect_df.apply(
    add_effect_direction, axis=1
)

print(
    f"Variants with an NGN2 readout: {variant_effect_df['NGN2_variant_has_readout'].sum()}"
)
print(variant_effect_df["NGN2_variant_effect_direction"].value_counts())

# focusing on the tested

variant_effect_df_tested = variant_effect_df.loc[variant_effect_df["ID"].str.contains("cardiac_neuro_cava_random")].copy()
print(
    f"Tested variants with an NGN2 readout: {variant_effect_df_tested['NGN2_variant_has_readout'].sum()}"
)
print(variant_effect_df_tested["NGN2_variant_effect_direction"].value_counts())


Variants with an NGN2 readout: 40450
NGN2_variant_effect_direction
not_significant    46950
upregulating         661
downregulating       562
Name: count, dtype: int64
Tested variants with an NGN2 readout: 38968
NGN2_variant_effect_direction
not_significant    45356
upregulating         605
downregulating       413
Name: count, dtype: int64


In [6]:
# variants whose sequence didn't resolve to genomic coordinates in metadata can't carry any
# of the coordinate-based annotations below -- drop them (matches the original notebook)
n_before = variant_effect_df.shape[0]
variant_effect_df = variant_effect_df.dropna(subset=["start", "end"]).copy()
variant_effect_df["start"] = variant_effect_df["start"].astype("Int64")
variant_effect_df["end"] = variant_effect_df["end"].astype("Int64")
print(f"Dropped {n_before - variant_effect_df.shape[0]} rows with no genomic coordinates")

# SPDI/variant_pos come out of metadata as single-element lists per variant -- collapse them
variant_effect_df[col_SPDI] = variant_effect_df[col_SPDI].apply(
    lambda x: x[0] if isinstance(x, list) else x
)
variant_effect_df[col_variant_pos] = variant_effect_df[col_variant_pos].apply(
    lambda x: x[0] if isinstance(x, list) else x
)
variant_effect_df = variant_effect_df.loc[variant_effect_df[col_SPDI].notna()].copy()

Dropped 90 rows with no genomic coordinates


### Regenerate variant position using alignment of reference and alternative allele sequences
The design-metadata `variant_pos` is not trusted (it does not consistently agree with the
actual REF/ALT sequence content); recompute it from the aligned sequences instead.

In [7]:
import difflib


def get_alignment_based_variant_pos(ref_seq, alt_seq):
    """0-based position of the first mismatching opcode block between ref_seq and alt_seq.
    Works cleanly for SNVs; for indels this is the start of the differing block, not
    necessarily a single base -- applied uniformly regardless of variant class, as in the
    original notebook."""
    seq_matcher = difflib.SequenceMatcher(None, ref_seq, alt_seq)
    for tag, i1, i2, j1, j2 in seq_matcher.get_opcodes():
        if tag != "equal":
            return i1
    raise ValueError("ref_seq and alt_seq are identical -- no variant position found")


variant_effect_df[col_variant_pos] = variant_effect_df.apply(
    lambda row: get_alignment_based_variant_pos(row["ref_seq"], row["alt_seq"]), axis=1
)

### Annotate the Enformer effect class
Enformer-prioritization batches oversampled high/low-effect extremes to build a stratified
test set (high effect / low effect / random); resolve per-variant class across batches.

In [8]:
def get_enformer_class_from_list(enformer_class_list):
    if not isinstance(enformer_class_list, list):
        raise ValueError("enformer_class_list should be a list")
    if len(enformer_class_list) == 0:
        raise ValueError("enformer_class_list is empty")
    if len(enformer_class_list) == 1:
        return enformer_class_list[0]
    if "enformer_random" in enformer_class_list:
        return "enformer_random"
    raise ValueError(f"Multiple non-random Enformer classes for one variant: {enformer_class_list}")


variant_effect_df_tested = variant_effect_df.loc[
    variant_effect_df["ID"].str.startswith("cardiac_neuro_cava_random:")
].copy()
variant_effect_df_tested["chr_pos_ref_alt"] = variant_effect_df_tested["ALT"].apply(
    hf.get_chrom_pos_ref_alt_pattern
)

path_to_enformer_files = "data/random_sampling"
all_enformer_prioritization_files = [
    os.path.join(path_to_enformer_files, file)
    for file in listdir(path_to_enformer_files)
    if file.startswith("2604_enformer_prio_class_")
]
all_enformer_prio_df = pd.concat(
    [pd.read_csv(file, sep="\t", low_memory=False) for file in all_enformer_prioritization_files]
)

resolved_df = (
    all_enformer_prio_df.groupby("chr_pos_ref_alt")["new_enformer_class"]
    .apply(lambda x: sorted(x.unique()))
    .reset_index()
    .rename(columns={"new_enformer_class": "enformer_classes_list"})
)
all_enformer_prio_df_deduplicated = (
    all_enformer_prio_df.merge(resolved_df, on="chr_pos_ref_alt", how="left")
    .drop(columns=["gene_set", "enformer_class", "new_enformer_class"])
    .drop_duplicates(subset="chr_pos_ref_alt", keep="first")
)
all_enformer_prio_df_deduplicated["enformer_class"] = all_enformer_prio_df_deduplicated[
    "enformer_classes_list"
].apply(lambda x: pd.NA if x != x else get_enformer_class_from_list(x))

interesting_columns_enformer = ["chr_pos_ref_alt", "DNase_max", "max_col", "enformer_classes_list", "enformer_class"]
variant_effect_df_tested_enformer = variant_effect_df_tested.merge(
    all_enformer_prio_df_deduplicated[interesting_columns_enformer], on="chr_pos_ref_alt", how="left"
)

variant_effect_df_not_tested = variant_effect_df.loc[
    ~variant_effect_df["ID"].str.startswith("cardiac_neuro_cava_random:")
].copy()
variant_effect_df = pd.concat(
    [variant_effect_df_tested_enformer, variant_effect_df_not_tested], ignore_index=True
)
print(f"Variants with an Enformer class: {variant_effect_df['enformer_class'].notna().sum()}")

if plotting:
    plt.figure(figsize=(8, 5))
    sns.histplot(
        data=variant_effect_df_tested_enformer, x="DNase_max", bins=50, kde=True, hue="enformer_class"
    )
    plt.title("Distribution of Enformer DNase Max Predictions")
    plt.show()

Variants with an Enformer class: 24373


In [9]:
# high-effect percentile flags, computed over all variants with a measured effect
variant_effect_df["NGN2_variant_abs_logFC"] = variant_effect_df["NGN2_variant_logFC"].abs()
thr_90 = variant_effect_df["NGN2_variant_abs_logFC"].quantile(0.90)
thr_95 = variant_effect_df["NGN2_variant_abs_logFC"].quantile(0.95)
variant_effect_df["NGN2_high_effect_var_09"] = variant_effect_df["NGN2_variant_abs_logFC"] >= thr_90
variant_effect_df["NGN2_high_effect_var_095"] = variant_effect_df["NGN2_variant_abs_logFC"] > thr_95
print(f"90th/95th percentile abs(logFC) thresholds: {thr_90:.3f} / {thr_95:.3f}")

90th/95th percentile abs(logFC) thresholds: 0.270 / 0.351


In [10]:
variant_effect_df_tested = variant_effect_df.loc[
    variant_effect_df["ID"].str.startswith("cardiac_neuro_cava_random:")
].copy()

print(
    f"Tested variants with an NGN2 readout: {variant_effect_df_tested['NGN2_variant_has_readout'].sum()}"
)
print(variant_effect_df_tested["NGN2_variant_effect_direction"].value_counts())


Tested variants with an NGN2 readout: 38968
NGN2_variant_effect_direction
not_significant    45356
upregulating         605
downregulating       413
Name: count, dtype: int64


### Annotate with gnomAD allele frequencies

In [11]:
# --- hg38 / GRCh38 RefSeq chromosome mapping ---
REFSEQ_CHROM_MAP_HG38 = {
    "1": "NC_000001.11", "2": "NC_000002.12", "3": "NC_000003.12", "4": "NC_000004.12",
    "5": "NC_000005.10", "6": "NC_000006.12", "7": "NC_000007.14", "8": "NC_000008.11",
    "9": "NC_000009.12", "10": "NC_000010.11", "11": "NC_000011.10", "12": "NC_000012.12",
    "13": "NC_000013.11", "14": "NC_000014.9", "15": "NC_000015.10", "16": "NC_000016.10",
    "17": "NC_000017.11", "18": "NC_000018.10", "19": "NC_000019.10", "20": "NC_000020.11",
    "21": "NC_000021.9", "22": "NC_000022.11", "X": "NC_000023.11", "Y": "NC_000024.10",
    "MT": "NC_012920.1", "M": "NC_012920.1",
}


def add_spdi_to_gnomad(df):
    """Build a hg38 RefSeq SPDI column on a gnomAD dataframe with gnomad_{chrom,pos,ref,alt}."""
    df = df.copy()
    required = ["gnomad_chrom", "gnomad_pos", "gnomad_ref", "gnomad_alt"]
    assert_condition(all(c in df.columns for c in required), "gnomAD dataframe missing required columns")

    df["gnomad_chrom"] = df["gnomad_chrom"].astype(str).str.replace("^chr", "", regex=True).str.upper()
    df["refseq_chrom"] = df["gnomad_chrom"].map(REFSEQ_CHROM_MAP_HG38)
    unmapped = df[df["refseq_chrom"].isna()]["gnomad_chrom"].unique()
    assert_condition(len(unmapped) == 0, f"Unmapped gnomAD chromosomes: {unmapped}")

    df["spdi_pos0"] = df["gnomad_pos"] - 1
    assert_condition((df["spdi_pos0"] >= 0).all(), "Negative SPDI coordinates detected!")

    df["gnomad_SPDI"] = (
        df["refseq_chrom"] + ":" + df["spdi_pos0"].astype(str) + ":" + df["gnomad_ref"] + ":" + df["gnomad_alt"]
    )
    return df


def merge_gnomad_into_variant_effect_df(variant_df, gnomad_df, gnomad_cols):
    """Left-join selected gnomAD AC/AF columns onto variant_df via SPDI, with hard guardrails."""
    gnomad_df = add_spdi_to_gnomad(gnomad_df)
    assert_condition(
        all(c in gnomad_df.columns for c in gnomad_cols), "Missing requested gnomAD columns"
    )
    gnomad_small = gnomad_df[["gnomad_SPDI"] + gnomad_cols].copy()
    assert_condition(
        gnomad_small["gnomad_SPDI"].is_unique, "Non-unique SPDI in gnomAD! Merging would duplicate rows."
    )

    merged = variant_df.merge(gnomad_small, left_on="SPDI", right_on="gnomad_SPDI", how="left")
    assert_condition(len(merged) == len(variant_df), "Row count changed after merge -- unexpected behavior.")

    print(f"Variants with gnomAD AF: {merged[gnomad_cols[0]].notna().sum()}")
    print(f"Variants missing gnomAD (expected for control sequences): {merged[gnomad_cols[0]].isna().sum()}")
    return merged


gnomad_data_all_variants = pd.read_csv(config["files"]["creating"]["gnomad_data_all_variants_local"], sep="\t")
gnomad_data_all_variants.columns = ["gnomad_" + col for col in gnomad_data_all_variants.columns]
gnomad_data_all_variants_unique = gnomad_data_all_variants.drop_duplicates(
    subset=["gnomad_chrom", "gnomad_pos", "gnomad_ref", "gnomad_alt"]
)

gnomad_cols_to_add = [
    "gnomad_AC", "gnomad_AF", "gnomad_AF_popmax", "gnomad_AF_eas",
    "gnomad_AF_nfe", "gnomad_AF_fin", "gnomad_AF_afr", "gnomad_AF_asj",
]
variant_effect_df = merge_gnomad_into_variant_effect_df(
    variant_effect_df, gnomad_data_all_variants_unique, gnomad_cols_to_add
)


def add_common_rare_singleton_category(af, ac):
    """
    - 'very common': af >= 0.05
    - 'common': af > 0.01
    - 'rare': af <= 0.01
    - 'singleton': ac == 1
    - 'unknown': no gnomAD entry (e.g. control sequences)
    """
    if pd.isna(af) or pd.isna(ac):
        return "unknown"
    if ac == 1:
        return "singleton"
    if af >= 0.05:
        return "very common"
    if af > 0.01:
        return "common"
    return "rare"


variant_effect_df["af_category"] = variant_effect_df.apply(
    lambda row: add_common_rare_singleton_category(row["gnomad_AF"], row["gnomad_AC"]), axis=1
)
print(variant_effect_df["af_category"].value_counts())

Variants with gnomAD AF: 46382
Variants missing gnomAD (expected for control sequences): 1701
af_category
rare           12236
very common    12153
singleton      12137
common          9856
unknown         1701
Name: count, dtype: int64


In [12]:
variant_effect_df_tested = variant_effect_df.loc[
    variant_effect_df["ID"].str.startswith("cardiac_neuro_cava_random:")
].copy()

print(
    f"Tested variants with an NGN2 readout: {variant_effect_df_tested['NGN2_variant_has_readout'].sum()}"
)
print(variant_effect_df_tested["NGN2_variant_effect_direction"].value_counts())


Tested variants with an NGN2 readout: 38968
NGN2_variant_effect_direction
not_significant    45356
upregulating         605
downregulating       413
Name: count, dtype: int64


### Investigate transition vs. transversion
Only the lightweight per-variant substitution classification is kept as a table column;
the Ti/Tv enrichment testing and AF-stratified plots in the original notebook are
exploratory manuscript-figure analysis (see `05_variant_transition_transversion.py`).

In [13]:
def classify_substitution(ref_seq, alt_seq, variant_pos):
    ref, alt = ref_seq[variant_pos], alt_seq[variant_pos]
    if ref == alt:
        raise ValueError("Reference and alternate alleles are the same.")
    purines, pyrimidines = {"A", "G"}, {"C", "T"}
    if (ref in purines and alt in purines) or (ref in pyrimidines and alt in pyrimidines):
        return "transition"
    return "transversion"


has_snv_alleles = (
    variant_effect_df["ref_seq"].notna()
    & variant_effect_df["alt_seq"].notna()
    & (variant_effect_df["ref_seq"].str.len() == variant_effect_df["alt_seq"].str.len())
)
variant_effect_df.loc[has_snv_alleles, "substitution_type"] = variant_effect_df.loc[has_snv_alleles].apply(
    lambda row: classify_substitution(row["ref_seq"], row["alt_seq"], row["variant_pos"]), axis=1
)
variant_effect_df["is_transversion"] = variant_effect_df["substitution_type"] == "transversion"
print(variant_effect_df["substitution_type"].value_counts())

substitution_type
transition      31228
transversion    16850
Name: count, dtype: int64


In [14]:
# has readout and is tested NGN2_variant_has_readout
variant_effect_df_raw = variant_effect_df.copy()
variant_effect_df = variant_effect_df.loc[variant_effect_df["NGN2_variant_has_readout"] & variant_effect_df["ID"].str.startswith("cardiac_neuro_cava_random:")].copy()

### Annotate with TFBS findings
FIMO/HOCOMOCO H13CORE PWM hits, scored for how well REF vs. ALT fit the motif (local ±2bp
window around the variant, and the full motif window), kept only where the motif fits well
(relative bit score > 0.75) and the variant has a measured NGN2 effect. Where a variant
overlaps several candidate motifs, the TF most highly expressed in brain tissue (HPA) is
kept as the representative hit.

In [16]:
# NOTE: This file is too large you have to regenerate it using fimo and hocomoco, install Fimo in e.g. a conda environment (conda install bioconda::meme) and download the hocomoco motif database
# See here: ./data/h13_fimo_overlap/README.md
# downloading h13 code meme format https://hocomoco14.autosome.org/final_bundle/hocomoco13/H13CORE/formatted_motifs/H13CORE_meme_format.meme (downloaded 16.05.2025)
dummy_subset_fimo_h13_tfbs_predictions_path = "./data/h13_fimo_overlap/head_100000_all_neuro_ctrls_scrambled_tested_h13.tsv.gz"
dummy = True
# generated via:
# fimo --max-stored-scores 1000000 --o /home/kisa/coding/80K_MPRA/fimo_data/element_tf_search/fimo_h13_clustered_all_useful_sequences_neuro_ctrls_scrambled_tested/all_neuro_ctrls_scrambled_tested_h13  /home/kisa/coding/80K_MPRA/fimo_data/Pia_generated_h13core/H13CORE_meme_format_clustered.meme /home/kisa/coding/80K_MPRA/fimo_data/element_tf_search/fimo_h13_clustered_all_useful_sequences_neuro_ctrls_scrambled_tested/80K_tested_neuro_ctrls_scrambled_fimo_input.fa

fimo_result_df = pd.read_csv(dummy_subset_fimo_h13_tfbs_predictions_path, sep="\t", comment="#")
if dummy:
    fimo_result_df = pd.read_csv(dummy_subset_fimo_h13_tfbs_predictions_path, sep="\t", comment="#")

fimo_result_df["sequence_name"] = fimo_result_df["sequence_name"].apply(
    lambda name: "cardiac_neuro_cava_random:" + name
)

important_cols_for_matching = ["ID", "REF", "ALT", "ref_seq", "alt_seq", "variant_pos"]
fimo_alt = fimo_result_df.merge(
    variant_effect_df[important_cols_for_matching], left_on="sequence_name", right_on="ALT", how="inner"
).rename(columns={"q-value": "ALT_q-value", "sequence_name": "ALT_name"})
fimo_ref = fimo_result_df.merge(
    variant_effect_df[important_cols_for_matching], left_on="sequence_name", right_on="REF", how="inner"
).rename(columns={"q-value": "REF_q-value", "sequence_name": "REF_name"})

combined_fimo_df = pd.merge(
    fimo_alt[["ID", "motif_id", "start", "stop", "strand", "ALT_name", "ALT_q-value"]],
    fimo_ref[["ID", "motif_id", "start", "stop", "strand", "REF_name", "REF_q-value"]],
    on=["ID", "motif_id", "start", "stop", "strand"],
    how="outer",
).rename(columns={"start": "fimo_start", "stop": "fimo_stop", "motif_id": "H13_motif_id", "strand": "fimo_strand"})

combined_fimo_df_variant_info = combined_fimo_df.merge(
    variant_effect_df[["ID", "ref_seq", "alt_seq", "variant_pos", "NGN2_variant_adj_p-value", "NGN2_variant_logFC"]],
    on="ID",
    how="inner",
)


def motif_overlaps_variant(row):
    """fimo_start/fimo_stop are 1-based inclusive; variant_pos is 0-based."""
    variant_pos1 = row["variant_pos"] + 1
    return row["fimo_start"] <= variant_pos1 <= row["fimo_stop"]


combined_fimo_df_overlapping_variants = combined_fimo_df_variant_info.loc[
    combined_fimo_df_variant_info.apply(motif_overlaps_variant, axis=1)
].copy()
print(
    f"Motif hits overlapping their variant: {combined_fimo_df_overlapping_variants.shape[0]}"
    f" ({combined_fimo_df_overlapping_variants['ID'].nunique()} unique variants)"
)

Motif hits overlapping their variant: 549 (290 unique variants)


In [18]:
meme_file = "data/h13_fimo_overlap/H13CORE_meme_format.meme"


def extract_motif_names(meme_file_path):
    motif_names = []
    with open(meme_file_path) as f:
        for line in f:
            if line.startswith("MOTIF"):
                motif_names.append(line.split(" ", 1)[1].strip())
    return motif_names


motif_names = extract_motif_names(meme_file)
hocomoco_motif_dict = {}
with open(meme_file) as handle:
    for i, motif in enumerate(motifs.parse(handle, "pfm-four-columns")):
        motif.name = motif_names[i]
        hocomoco_motif_dict[motif.name] = motif


def reverse_complement(seq):
    complement = str.maketrans("ACGTacgt", "TGCAtgca")
    return seq.translate(complement)[::-1]


def local_bitscore_tfbs(motif, sequence, motif_start, motif_end, variant_pos, strand):
    """Local bit score: PWM probability fit summed over a 5bp window (±2) centered on the
    variant position, normalized by the best-possible score for that same cropped window.
    `motif_start` is 1-based, `motif_end` is 1-based inclusive; `variant_pos` is 0-based."""
    zero_based_motif_start = motif_start - 1
    pwm = motif.pwm
    window = 5

    sequence_of_interest_start = max(zero_based_motif_start, variant_pos - math.floor(window / 2))
    sequence_of_interest_end = min(motif_end, variant_pos + math.floor(window / 2))

    # if the variant sits within window/2 of a motif edge, shift the window fully inside the motif
    if (variant_pos - zero_based_motif_start) < math.floor(window / 2):
        sequence_of_interest_start = zero_based_motif_start
        sequence_of_interest_end = zero_based_motif_start + window
    elif (motif_end - variant_pos - 1) < math.floor(window / 2):
        sequence_of_interest_end = motif_end
        sequence_of_interest_start = motif_end - window

    motif_related_start = max(0, sequence_of_interest_start - zero_based_motif_start)
    motif_related_end = min(motif_end - zero_based_motif_start, sequence_of_interest_end - zero_based_motif_start)

    motif_sequence = sequence[zero_based_motif_start:motif_end]
    if strand == "-":
        motif_sequence = reverse_complement(motif_sequence)
        motif_related_start, motif_related_end = (
            len(motif_sequence) - motif_related_end,
            len(motif_sequence) - motif_related_start,
        )

    motif_part = motif_sequence[motif_related_start:motif_related_end]
    modified_pwm = {nuc: scores[motif_related_start:motif_related_end] for nuc, scores in pwm.items()}

    best_motif_score = sum(
        max(modified_pwm[nuc][i] for nuc in BASES) for i in range(motif_related_end - motif_related_start)
    )
    local_seq_fit_score = sum(modified_pwm[nuc][i] for i, nuc in enumerate(motif_part))
    relative_local_fit_score = local_seq_fit_score / best_motif_score if best_motif_score != 0 else 0

    return local_seq_fit_score, relative_local_fit_score


def compute_ref_alt_scores(row):
    ref = local_bitscore_tfbs(
        hocomoco_motif_dict[row["H13_motif_id"]], row["ref_seq"], row["fimo_start"], row["fimo_stop"],
        row["variant_pos"], row["fimo_strand"],
    )
    alt = local_bitscore_tfbs(
        hocomoco_motif_dict[row["H13_motif_id"]], row["alt_seq"], row["fimo_start"], row["fimo_stop"],
        row["variant_pos"], row["fimo_strand"],
    )
    return pd.Series(
        {
            "REF_local_bit_score": ref[0], "REF_local_relative_bit_score": ref[1],
            "ALT_local_bit_score": alt[0], "ALT_local_relative_bit_score": alt[1],
        }
    )


def build_motif_cache(hocomoco_motif_dict):
    """Per-motif cached PWM + per-position/overall best-possible score, for the 'full motif
    width' scorer below."""
    motif_cache = {}
    for motif_id, motif in hocomoco_motif_dict.items():
        pwm = motif.pwm
        pos_max = np.array([max(pwm[nuc][i] for nuc in BASES) for i in range(pwm.length)])
        motif_cache[motif_id] = {"pwm": pwm, "length": pwm.length, "max_score": pos_max.sum()}
    return motif_cache


def full_relative_pwm_fit_cached(motif_id, sequence, motif_start, motif_end, strand, motif_cache):
    """PWM probability fit summed over the *entire* motif width (companion metric to the
    local ±2bp score above, not a competing method -- both are kept in the final table)."""
    motif_seq = sequence[motif_start - 1:motif_end]
    if strand == "-":
        motif_seq = reverse_complement(motif_seq)

    cache = motif_cache[motif_id]
    pwm, max_score = cache["pwm"], cache["max_score"]
    obs_score = sum(pwm[nuc][i] for i, nuc in enumerate(motif_seq))
    rel_score = obs_score / max_score if max_score > 0 else 0.0
    return obs_score, rel_score


def compute_full_ref_alt_scores(row, motif_cache):
    ref = full_relative_pwm_fit_cached(
        row["H13_motif_id"], row["ref_seq"], row["fimo_start"], row["fimo_stop"], row["fimo_strand"], motif_cache
    )
    alt = full_relative_pwm_fit_cached(
        row["H13_motif_id"], row["alt_seq"], row["fimo_start"], row["fimo_stop"], row["fimo_strand"], motif_cache
    )
    return pd.Series(
        {
            "REF_full_bit_score": ref[0], "REF_full_relative_bit_score": ref[1],
            "ALT_full_bit_score": alt[0], "ALT_full_relative_bit_score": alt[1],
        }
    )


combined_fimo_df_overlapping_variants = combined_fimo_df_overlapping_variants.join(
    combined_fimo_df_overlapping_variants.apply(compute_ref_alt_scores, axis=1)
)
motif_cache = build_motif_cache(hocomoco_motif_dict)
combined_fimo_df_overlapping_variants = combined_fimo_df_overlapping_variants.join(
    combined_fimo_df_overlapping_variants.apply(lambda row: compute_full_ref_alt_scores(row, motif_cache), axis=1)
)

combined_fimo_df_overlapping_variants["max_local_bit_score"] = combined_fimo_df_overlapping_variants[
    ["REF_local_bit_score", "ALT_local_bit_score"]
].max(axis=1)
combined_fimo_df_overlapping_variants["max_local_relative_bit_score"] = combined_fimo_df_overlapping_variants[
    ["REF_local_relative_bit_score", "ALT_local_relative_bit_score"]
].max(axis=1)
combined_fimo_df_overlapping_variants["max_full_bit_score"] = combined_fimo_df_overlapping_variants[
    ["REF_full_bit_score", "ALT_full_bit_score"]
].max(axis=1)
combined_fimo_df_overlapping_variants["max_full_relative_bit_score"] = combined_fimo_df_overlapping_variants[
    ["REF_full_relative_bit_score", "ALT_full_relative_bit_score"]
].max(axis=1)
combined_fimo_df_overlapping_variants["delta_local_bit_score"] = (
    combined_fimo_df_overlapping_variants["ALT_local_bit_score"] - combined_fimo_df_overlapping_variants["REF_local_bit_score"]
)
combined_fimo_df_overlapping_variants["abs_delta_local_bit_score"] = combined_fimo_df_overlapping_variants[
    "delta_local_bit_score"
].abs()

In [ ]:
# keep only well-fitting motif hits on variants with a measured effect
combined_fimo_df_overlapping_variants_filtered = combined_fimo_df_overlapping_variants.loc[
    (combined_fimo_df_overlapping_variants["max_local_relative_bit_score"] > relative_bitscore_threshold)
    & (combined_fimo_df_overlapping_variants["NGN2_variant_logFC"].notna())
].copy()
print(
    f"After filtering to relative local bit score > {relative_bitscore_threshold}: "
    f"{combined_fimo_df_overlapping_variants_filtered['ID'].nunique()} unique variants"
)

# map H13 motif id -> TF gene symbol
annotation_file = "data/h13_fimo_overlap/H13CORE_annotation.jsonl"
with open(annotation_file) as f:
    anno_df = pd.DataFrame(json.loads(line) for line in f if line.strip())


def extract_gene_symbol_from_masterlist(info):
    if pd.isna(info):
        return None
    if isinstance(info, str):
        try:
            info = json.loads(info)
        except json.JSONDecodeError:
            try:
                info = ast.literal_eval(info)
            except Exception:
                return None
    if not isinstance(info, dict):
        return None
    gene_symbol = info.get("species", {}).get("HUMAN", {}).get("gene_symbol")
    if gene_symbol:
        return gene_symbol
    return info.get("tf")


anno_df["tf_gene_symbol"] = anno_df["masterlist_info"].apply(extract_gene_symbol_from_masterlist)
motif_to_tf_df = anno_df[["name", "tf_gene_symbol"]].drop_duplicates().rename(columns={"name": "H13_motif_id"})
combined_fimo_df_overlapping_variants_filtered = combined_fimo_df_overlapping_variants_filtered.merge(
    motif_to_tf_df, on="H13_motif_id", how="left"
)

# disambiguate multiple candidate TFs per variant by picking the one most highly expressed
# in brain tissue (HPA), using the max nTPM across all brain regions (not just cortex)
hpa_brain_df = pd.read_csv("../cCRE_based_analysis/data/rna_brain_region_hpa.tsv.gz", sep="\t")
brain_region_nTPMs = hpa_brain_df.groupby("Gene name")["nTPM"].max().reset_index()
combined_fimo_df_overlapping_variants_filtered = combined_fimo_df_overlapping_variants_filtered.merge(
    brain_region_nTPMs, left_on="tf_gene_symbol", right_on="Gene name", how="left"
)

combined_fimo_df_overlapping_variants_expression = (
    combined_fimo_df_overlapping_variants_filtered.sort_values(by="nTPM", ascending=False)
    .drop_duplicates(subset=["ID"], keep="first")
    .rename(columns={"nTPM": "tf_nTPM_brain"})
)
combined_fimo_df_overlapping_variants_expression["has_h13_tfbs_annotation"] = (
    combined_fimo_df_overlapping_variants_expression["H13_motif_id"].notna()
)

n_unique_variants = combined_fimo_df_overlapping_variants_expression["ID"].nunique()
n_unique_tfs = combined_fimo_df_overlapping_variants_expression["tf_gene_symbol"].nunique()
print(f"Number of unique variants with a TFBS annotation: {n_unique_variants}")
print(f"Number of unique TFs represented: {n_unique_tfs}")

After filtering to relative local bit score > 0.75: 273 unique variants
Number of unique variants with a TFBS annotation: 273
Number of unique TFs represented: 32


In [20]:
interesting_columns_for_variants_table = [
    "ID", "H13_motif_id", "tf_gene_symbol", "tf_nTPM_brain", "fimo_start", "fimo_stop", "fimo_strand",
    "REF_local_bit_score", "REF_local_relative_bit_score", "ALT_local_bit_score", "ALT_local_relative_bit_score",
    "REF_full_bit_score", "REF_full_relative_bit_score", "ALT_full_bit_score", "ALT_full_relative_bit_score",
    "max_local_bit_score", "max_local_relative_bit_score", "max_full_bit_score", "max_full_relative_bit_score",
    "delta_local_bit_score", "abs_delta_local_bit_score", "has_h13_tfbs_annotation",
]
variant_effect_df = variant_effect_df.merge(
    combined_fimo_df_overlapping_variants_expression[interesting_columns_for_variants_table], on="ID", how="left"
)
variant_effect_df["has_h13_tfbs_annotation"] = variant_effect_df["has_h13_tfbs_annotation"].fillna(False)

tfbs_checkpoint_path = "data/variant_overlapping_tfbs_scored_bit_score_and_relative_bit_score_local_g075.tsv.gz"
if writing:
    combined_fimo_df_overlapping_variants_expression[interesting_columns_for_variants_table].to_csv(
        tfbs_checkpoint_path, sep="\t", index=False, compression="gzip"
    )
    print(f"Wrote per-variant TFBS annotation checkpoint to {tfbs_checkpoint_path}")

## Annotate with element-based annotations
Pull in gene_set / TSS distance / E2G / SCREEN / region-level eQTL / phastCons / singleton
density from the already-cleaned element annotation table
(`element_annotation/generate_element_annotation_table.py`), plus the reference and
alternate allele's own element activity.

In [21]:
element_based_annotation_path = "../cCRE_based_analysis/data/all_80k_MPRA_elements_annotations_table_2606_eQTL_phastCons_ReMap.tsv.gz"

element_based_annotation_df = pd.read_csv(element_based_annotation_path, sep="\t")
for col in ["gene_set", "REMAP2022_TFBS_list"]:
    if col in element_based_annotation_df.columns:
        element_based_annotation_df[col] = element_based_annotation_df[col].apply(hf.safe_eval)
element_based_annotation_df_with_regions = element_based_annotation_df.loc[
    element_based_annotation_df["start"].notna()
].copy()

/tmp/ipykernel_738637/1560788540.py:3: DtypeWarning: Columns (0: source, 1: variant_class, 2: variant_pos, 3: SPDI, 4: allele, 5: info, 6: brain_eQTL_gene_symbols) have mixed types. Specify dtype option on import or set low_memory=False.
  element_based_annotation_df = pd.read_csv(element_based_annotation_path, sep="\t")


In [22]:
# gene_set / associated_gene_name from the variant's own header, independent of the element table
gene_list_dir = "../cCRE_based_analysis/data/gene_lists"
gene_lists = [f for f in os.listdir(gene_list_dir) if f.endswith(".txt")]


def get_gene_lookup_dict(gene_lists, gene_list_dir):
    gene_lookup_dict = defaultdict(list)
    for file in gene_lists:
        with open(os.path.join(gene_list_dir, file)) as f:
            gene_list_name = file.split(".")[0]
            for gene in f.read().splitlines():
                gene_lookup_dict[gene].append(gene_list_name)
    return gene_lookup_dict


gene_lookup_dict = get_gene_lookup_dict(gene_lists, gene_list_dir)
variant_effect_df["associated_gene_name"] = variant_effect_df["ALT"].apply(hf.get_gene_name)
variant_effect_df["gene_set"] = variant_effect_df["associated_gene_name"].apply(lambda gene: gene_lookup_dict[gene])
print(variant_effect_df["gene_set"].apply(tuple).value_counts())

gene_set
(neuro,)                  16107
(cardiac,)                11166
(cava,)                    6130
(random,)                  3606
(neuro, cardiac)           1130
(neuro, cava)               452
(cava, cardiac)             314
(neuro, cava, cardiac)       63
Name: count, dtype: int64


In [23]:
# TSS distance to the associated gene (GENCODE v42), nearest transcript TSS *of that gene only*
# NOTE too big to store in github: download and store in data/ https://www.gencodegenes.org/human/release_42.html
gene_annotation_path = "data/gencode.v42.gtf.gz"
gene_annotations_df = pd.read_csv(gene_annotation_path, sep="\t", comment="#", header=None)
gene_annotations_df.columns = ["chrom", "source", "feature", "start", "end", "score", "strand", "frame", "attribute"]
gene_only_transcripts_df = gene_annotations_df.loc[gene_annotations_df["feature"] == "transcript"].copy()


def extract_gtf_field(attribute, field):
    for item in attribute.split(";"):
        item = item.strip()
        if item.startswith(field):
            return item.split(" ")[1].replace('"', "")
    return None


gene_only_transcripts_df["gene_id"] = gene_only_transcripts_df["attribute"].apply(
    lambda a: extract_gtf_field(a, "gene_id")
)
gene_only_transcripts_df["gene_name"] = gene_only_transcripts_df["attribute"].apply(
    lambda a: extract_gtf_field(a, "gene_name")
)
gene_only_transcripts_df["tss"] = gene_only_transcripts_df.apply(
    lambda r: r["start"] if r["strand"] == "+" else r["end"], axis=1
)

tss_df = gene_only_transcripts_df.copy()
tss_df["Start"] = tss_df["tss"] - 1
tss_df["End"] = tss_df["tss"]
tss_df["Chromosome"] = tss_df["chrom"]
tss_pr = pr.PyRanges(
    tss_df[["Chromosome", "Start", "End", "strand", "gene_name", "gene_id"]].rename(columns={"strand": "Strand"})
)

var_df = variant_effect_df.copy()
var_df["genomic_variant_pos_hg38"] = var_df["SPDI"].str.split(":").str[1].astype(int)
var_df["Chromosome"] = var_df["chr"]
var_df["Start"] = var_df["genomic_variant_pos_hg38"]
var_df["End"] = var_df["genomic_variant_pos_hg38"] + 1
var_pr = pr.PyRanges(
    var_df[["Chromosome", "Start", "End", "SPDI", "associated_gene_name"]].rename(
        columns={"associated_gene_name": "gene_name"}
    )
)

tss_results = []
for gene, v_sub in var_pr.df.groupby("gene_name"):
    t_sub = tss_pr.df.loc[tss_pr.df["gene_name"] == gene]
    if t_sub.empty:
        continue
    nearest = pr.PyRanges(v_sub).nearest(pr.PyRanges(t_sub), how="nearest")
    tss_results.append(nearest.df)

variant_with_gene_distance = pd.concat(tss_results, ignore_index=True)
assert_condition(
    (variant_with_gene_distance["gene_name"] == variant_with_gene_distance["gene_name_b"]).all(),
    "Gene names do not match between variant and TSS annotation",
)
variant_with_gene_distance = variant_with_gene_distance.rename(
    columns={"Distance": "minimal_tss_distance_associated_gene", "gene_name_b": "closest_tss_gene_name",
             "Start_b": "closest_tss_0based", "Strand": "closest_tss_strand"}
).drop_duplicates(subset=["SPDI"])

interesting_columns_tss_distance = [
    "SPDI", "minimal_tss_distance_associated_gene", "closest_tss_gene_name", "closest_tss_0based", "closest_tss_strand",
]

variant_effect_df = variant_effect_df.merge(
    variant_with_gene_distance[interesting_columns_tss_distance], on="SPDI", how="left"
)
print(f"Variants with a TSS distance: {variant_effect_df['minimal_tss_distance_associated_gene'].notna().sum()}")


Variants with a TSS distance: 38968


In [24]:
variant_effect_df.columns.to_list()

['ID',
 'REF',
 'ALT',
 'new_REF',
 'new_ALT',
 'ref_seq',
 'chr',
 'start',
 'end',
 'strand',
 'alt_seq',
 'variant_pos',
 'SPDI',
 'NGN2_variant_adj_p-value',
 'NGN2_variant_logFC',
 'NGN2_variant_has_readout',
 'NGN2_variant_is_significant',
 'NGN2_variant_effect_direction',
 'chr_pos_ref_alt',
 'DNase_max',
 'max_col',
 'enformer_classes_list',
 'enformer_class',
 'NGN2_variant_abs_logFC',
 'NGN2_high_effect_var_09',
 'NGN2_high_effect_var_095',
 'gnomad_SPDI',
 'gnomad_AC',
 'gnomad_AF',
 'gnomad_AF_popmax',
 'gnomad_AF_eas',
 'gnomad_AF_nfe',
 'gnomad_AF_fin',
 'gnomad_AF_afr',
 'gnomad_AF_asj',
 'af_category',
 'substitution_type',
 'is_transversion',
 'H13_motif_id',
 'tf_gene_symbol',
 'tf_nTPM_brain',
 'fimo_start',
 'fimo_stop',
 'fimo_strand',
 'REF_local_bit_score',
 'REF_local_relative_bit_score',
 'ALT_local_bit_score',
 'ALT_local_relative_bit_score',
 'REF_full_bit_score',
 'REF_full_relative_bit_score',
 'ALT_full_bit_score',
 'ALT_full_relative_bit_score',
 'max_loc

In [25]:
variant_effect_df_tmp = variant_effect_df.copy()

In [26]:
# reference vs. alternate allele's own element (region) activity + emVar flag
activity_info_cols = ["logFC", "adj.P.Val", "significant_region_NGN2", "normalized_activity_NGN2"]
activity_info_df = element_based_annotation_df_with_regions[["sequence"] + activity_info_cols].rename(
    columns={"logFC": "NGN2_region_logFC", "adj.P.Val": "NGN2_region_adj_P_Val", "normalized_activity_NGN2": "NGN2_region_normalized_activity"}
)

ref_activity_info_df = activity_info_df.rename(
    columns={
        "sequence": "ref_seq", "NGN2_region_logFC": "NGN2_ref_region_logFC",
        "NGN2_region_adj_P_Val": "NGN2_ref_region_adj_P_Val", "NGN2_region_normalized_activity": "NGN2_ref_region_normalized_activity",
        "significant_region_NGN2": "NGN2_ref_significant_region",
    }
)
alt_activity_info_df = activity_info_df.rename(
    columns={
        "sequence": "alt_seq", "NGN2_region_logFC": "NGN2_alt_region_logFC",
        "NGN2_region_adj_P_Val": "NGN2_alt_region_adj_P_Val", "NGN2_region_normalized_activity": "NGN2_alt_region_normalized_activity",
        "significant_region_NGN2": "NGN2_alt_significant_region",
    }
)

variant_effect_df = variant_effect_df.merge(ref_activity_info_df, on="ref_seq", how="left")
variant_effect_df = variant_effect_df.merge(alt_activity_info_df, on="alt_seq", how="left")

# emVar: a significant allelic variant where at least one allele's own element is itself a
# significantly active region
variant_effect_df["is_emVar"] = (variant_effect_df["NGN2_variant_adj_p-value"] < significant_threshold) & (
    variant_effect_df["NGN2_ref_significant_region"] | variant_effect_df["NGN2_alt_significant_region"]
)
print(variant_effect_df["is_emVar"].value_counts())

is_emVar
False    38613
True       355
Name: count, dtype: int64


In [27]:
element_based_annotation_df_with_regions.columns.to_list()

['name',
 'sequence',
 'category',
 'class',
 'source',
 'ref',
 'chr',
 'start',
 'end',
 'strand',
 'variant_class',
 'variant_pos',
 'SPDI',
 'allele',
 'info',
 'normalized_name',
 'logFC',
 'adj.P.Val',
 'normalized_activity_NGN2',
 'significant_region_NGN2',
 'effect_direction_NGN2',
 'region_info',
 'brain_e2g_cell_type',
 'brain_e2g_gene_name',
 'brain_e2g_score',
 'brain_e2g_distance_to_TSS',
 'cardiac_e2g_cell_type',
 'cardiac_e2g_gene_name',
 'cardiac_e2g_score',
 'cardiac_e2g_distance_to_TSS',
 'gene_id',
 'closest_tss_gene_name',
 'tss_distance',
 'gene_name',
 'gene_set',
 'SCREEN_ID',
 'is_ngn2_atac_overlap',
 'is_brain_SCREEN_cCRE',
 'is_heart_SCREEN_cCRE',
 'is_H1_SCREEN_cCRE',
 'dELS_is_brain_SCREEN_cCRE_group',
 'CA-only_is_brain_SCREEN_cCRE_group',
 'PLS_is_brain_SCREEN_cCRE_group',
 'pELS_is_brain_SCREEN_cCRE_group',
 'CA-H3K4me3_is_brain_SCREEN_cCRE_group',
 'CA-TF_is_brain_SCREEN_cCRE_group',
 'CA-only_is_heart_SCREEN_cCRE_group',
 'PLS_is_heart_SCREEN_cCRE_group

In [28]:
# region-level element annotations: E2G, SCREEN cCRE, region eQTL, phastCons, singleton density
region_info_cols = [
    "gene_set", "SCREEN_ID", "region_info", "brain_e2g_cell_type", "brain_e2g_gene_name", "brain_e2g_score", "brain_e2g_distance_to_TSS",
    "cardiac_e2g_cell_type", "cardiac_e2g_gene_name", "cardiac_e2g_score", "cardiac_e2g_distance_to_TSS",
    "is_brain_SCREEN_cCRE", "is_heart_SCREEN_cCRE", "is_H1_SCREEN_cCRE", "is_ngn2_atac_overlap",
    "closest_tss_gene_name", "tss_distance",
    "brain_eQTL_gene_symbols", "brain_eQTL_n_eqtls",
    "brain_eQTL_mean_beta",
    "brain_eQTL_median_beta",
    "brain_eQTL_sum_beta",
    "brain_eQTL_pip_weighted_beta",
    "brain_eQTL_fraction_positive",
    "has_overlapping_brain_eQTL",
    "phastCons_sum", "phastCons_mean0",
    "phastCons_mean", "singleton_overlap_count",
]
region_info_df = element_based_annotation_df_with_regions[["sequence"] + region_info_cols].rename(
    columns={
        "sequence": "ref_seq",
        # renamed to avoid colliding with the variant-level columns of the same name computed above
        "gene_set": "region_gene_set",
        "closest_tss_gene_name": "region_closest_tss_gene_name",
        "tss_distance": "region_tss_distance_to_closest_gene",
        "phastCons_sum": "region_phastCons_sum",
        "phastCons_mean0": "region_phastCons_mean0",
        "phastCons_mean": "region_phastCons_mean",
        "singleton_overlap_count": "region_singleton_overlap_count",
        "brain_eQTL_gene_symbols": "region_brain_eQTL_gene_symbols",
        "brain_eQTL_n_eqtls": "region_brain_eQTL_n_eqtls",
        "brain_eQTL_mean_beta": "region_brain_eQTL_mean_beta",
        "brain_eQTL_median_beta": "region_brain_eQTL_median_beta",
        "brain_eQTL_sum_beta": "region_brain_eQTL_sum_beta",
        "brain_eQTL_pip_weighted_beta": "region_brain_eQTL_pip_weighted_beta",
        "brain_eQTL_fraction_positive": "region_brain_eQTL_fraction_positive",
        "has_overlapping_brain_eQTL": "region_has_overlapping_brain_eQTL",
    }
)
variant_effect_df = variant_effect_df.merge(region_info_df, on="ref_seq", how="left")

## Annotate variants exactly with eQTLs (not just at region level)
GTEx / EMS / UKBB fine-mapping results, joined on the exact variant position rather than
the element's region -- built via a strand-aware canonical `chrom:pos:ref:alt` key.

In [29]:
def get_strand_aware_position(row):
    """1-based genomic position of the variant, counted from the correct end for '-' strand oligos."""
    if row["strand"] == "+":
        return row["start"] + row["variant_pos"] + 1
    return row["start"] + (len(row["ref_seq"]) - row["variant_pos"] - 1) + 1


def get_strand_aware_alleles(row):
    ref_allele, alt_allele = row["ref_seq"][row["variant_pos"]], row["alt_seq"][row["variant_pos"]]
    if row["strand"] == "-":
        ref_allele = str(Seq(ref_allele).reverse_complement())
        alt_allele = str(Seq(alt_allele).reverse_complement())
    return pd.Series([ref_allele, alt_allele])


mpra_variants = variant_effect_df.copy()
mpra_variants["pos"] = mpra_variants.apply(get_strand_aware_position, axis=1)
mpra_variants[["ref_allele", "alt_allele"]] = mpra_variants.apply(get_strand_aware_alleles, axis=1)
mpra_variants["var_key"] = (
    mpra_variants["chr"].astype(str) + ":" + mpra_variants["pos"].astype(str) + ":"
    + mpra_variants["ref_allele"] + ":" + mpra_variants["alt_allele"]
)
assert_condition((mpra_variants["ref_allele"].str.len() == 1).all(), "Non-SNV ref_allele in var_key construction")
assert_condition((mpra_variants["alt_allele"].str.len() == 1).all(), "Non-SNV alt_allele in var_key construction")
mpra_keys = set(mpra_variants["var_key"].unique())

In [30]:
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

def ensembl_batch_lookup(ensembl_ids, batch_size=200, sleep_time=0.5):
    """Batch ENSG -> gene symbol lookup via the Ensembl REST API."""
    url = "https://rest.ensembl.org/lookup/id"
    mapping = {}

    # session with automatic retries on transient errors
    session = requests.Session()
    retries = Retry(total=3, backoff_factor=1, status_forcelist=[429, 500, 502, 503, 504])
    session.mount("https://", HTTPAdapter(max_retries=retries))

    for i in range(0, len(ensembl_ids), batch_size):
        batch = ensembl_ids[i : i + batch_size]
        try:
            r = session.post(
                url,
                headers={"Content-Type": "application/json"},
                json={"ids": batch},
                timeout=(10, 120),  # (connect_timeout, read_timeout)
            )
            if not r.ok:
                print(f"Warning: Ensembl lookup failed for batch {i}-{i + batch_size} (HTTP {r.status_code})")
                continue
            for ens_id, info in r.json().items():
                mapping[ens_id] = info.get("display_name") if isinstance(info, dict) else None
        except requests.exceptions.RequestException as e:
            print(f"Warning: Ensembl lookup error for batch {i}-{i + batch_size}: {e}")
            continue
        time.sleep(sleep_time)
        print(f"  Looked up {min(i + batch_size, len(ensembl_ids))}/{len(ensembl_ids)} IDs")

    return mapping

In [31]:
# GTEx v8 (49 tissues, SuSiE fine-mapping) -- prefiltered to MPRA variants on the cluster via
# a companion script (`add_var_key_gtex.py`) since the raw release is too large to load
# directly; this only reads that prefiltered result.
gtex_prefiltered_path = "data/gtex_prefiltered_by_mpra_1_based.tsv.gz"
filtered_gtex_df = pd.read_csv(gtex_prefiltered_path, sep="\t", compression="gzip")
print(f"GTEx-matched variants: {filtered_gtex_df['var_key'].nunique()}")


filtered_gtex_df["ensembl_base"] = filtered_gtex_df["gene"].str.replace(r"\.\d+$", "", regex=True)
ensg_mapping = ensembl_batch_lookup(filtered_gtex_df["ensembl_base"].unique().tolist())
filtered_gtex_df["gene_symbol"] = filtered_gtex_df["ensembl_base"].map(ensg_mapping)
# a handful of retired Ensembl IDs return no display name -- drop those rows only
filtered_gtex_df = filtered_gtex_df.loc[filtered_gtex_df["gene_symbol"].notna()].copy()


def summarize_gtex_per_variant(df):
    summaries = []
    for var_key, sub in df.groupby("var_key"):
        max_pip = sub["pip"].max()
        best = sub.loc[sub["pip"] == max_pip].iloc[0]
        summaries.append(
            {
                "var_key": var_key,
                "gtex_eQTL_overlap": True,
                "gtex_tissues": ";".join(sub["tissue"].unique()),
                "gtex_n_tissues": sub["tissue"].nunique(),
                "gtex_genes": ";".join(sub["gene_symbol"].dropna().unique()),
                "gtex_n_genes": sub["gene_symbol"].dropna().nunique(),
                "gtex_best_tissue": best["tissue"],
                "gtex_best_gene": best["gene_symbol"],
                "gtex_max_pip": max_pip,
                "gtex_beta_at_best_pip": best["beta_marginal"],
                "gtex_beta_posterior_at_best_pip": best["beta_posterior"],
                "gtex_sd_posterior_at_best_pip": best["sd_posterior"],
                "gtex_in_credible_set": (sub["cs_id"] != -1).any(),
                "gtex_credible_set_ids": sub.loc[sub["cs_id"] != -1, "cs_id"].unique().tolist(),
            }
        )
    return pd.DataFrame(summaries)


gtex_summary_df = summarize_gtex_per_variant(filtered_gtex_df)
mpra_variants = mpra_variants.merge(gtex_summary_df, on="var_key", how="left")
mpra_variants["gtex_eQTL_overlap"] = mpra_variants["gtex_eQTL_overlap"].fillna(False)
mpra_variants["gtex_in_credible_set"] = mpra_variants["gtex_in_credible_set"].fillna(False)
# high-confidence / brain-specific overlap flags, derived from the summarized columns above
mpra_variants["gtex_high_confidence_eQTL_overlap"] = (mpra_variants["gtex_max_pip"] >= 0.5) & (
    mpra_variants["gtex_beta_posterior_at_best_pip"].abs() >= 0.2
)
mpra_variants["gtex_brain_eQTL_overlap"] = mpra_variants["gtex_tissues"].str.contains("brain", case=False, na=False)
mpra_variants["gtex_brain_eQTL_greater_01_beta_overlap"] = mpra_variants["gtex_brain_eQTL_overlap"] & (
    mpra_variants["gtex_beta_posterior_at_best_pip"].abs() > 0.1
)
mpra_variants["gtex_brain_high_confidence_eQTL_overlap"] = (
    mpra_variants["gtex_brain_eQTL_overlap"] & mpra_variants["gtex_high_confidence_eQTL_overlap"]
)

GTEx-matched variants: 10486
  Looked up 200/7155 IDs
  Looked up 400/7155 IDs
  Looked up 600/7155 IDs
  Looked up 800/7155 IDs
  Looked up 1000/7155 IDs
  Looked up 1200/7155 IDs
  Looked up 1400/7155 IDs
  Looked up 1600/7155 IDs
  Looked up 1800/7155 IDs
  Looked up 2000/7155 IDs
  Looked up 2200/7155 IDs
  Looked up 2400/7155 IDs
  Looked up 2600/7155 IDs
  Looked up 2800/7155 IDs
  Looked up 3000/7155 IDs
  Looked up 3200/7155 IDs
  Looked up 3400/7155 IDs
  Looked up 3600/7155 IDs
  Looked up 3800/7155 IDs
  Looked up 4000/7155 IDs
  Looked up 4200/7155 IDs
  Looked up 4400/7155 IDs
  Looked up 4600/7155 IDs
  Looked up 4800/7155 IDs
  Looked up 5000/7155 IDs
  Looked up 5200/7155 IDs
  Looked up 5400/7155 IDs
  Looked up 5600/7155 IDs
  Looked up 5800/7155 IDs
  Looked up 6000/7155 IDs
  Looked up 6200/7155 IDs
  Looked up 6400/7155 IDs
  Looked up 6600/7155 IDs
  Looked up 6800/7155 IDs
  Looked up 7000/7155 IDs
  Looked up 7155/7155 IDs


In [32]:
variant_effects_element_checkpoint_path = "data/variant_effects_with_gnomad_tfbs_element_annotations_all_gtex.tsv.gz"
if writing:
    mpra_variants.to_csv(variant_effects_element_checkpoint_path, sep="\t", index=False, compression="gzip")
    print(f"Wrote checkpoint (post gtex) to {variant_effects_element_checkpoint_path}")

In [33]:
# EMS (Expression Modifier Score), brain tissues only
def parse_ems_variant(v):
    """'chr10_100006504_T_C_b38' -> 'chr10:100006504:T:C'"""
    v = re.sub(r"_b38$", "", str(v).strip())
    chrom, pos, ref, alt = v.split("_")
    return f"{chrom}:{pos}:{ref}:{alt}"


ems_brain_files = sorted(glob.glob("data/ems_public/ems_top_Brain_*.tsv.bgz"))
ems_per_tissue = []
for ems_file in ems_brain_files:
    tissue = ems_file.split("/")[-1].replace("ems_top_", "").replace(".tsv.bgz", "")
    ems_tissue_df = pd.read_csv(ems_file, sep="\t", compression="gzip")
    ems_tissue_df["var_key"] = ems_tissue_df["v"].map(parse_ems_variant)
    ems_tissue_df["ensembl_base"] = ems_tissue_df["g"].str.replace(r"\.\d+$", "", regex=True)
    ems_tissue_summary = (
        ems_tissue_df.groupby("var_key")
        .agg(max_ems_norm=("ems_normalized", "max"), genes=("ensembl_base", lambda x: ";".join(sorted(set(x)))))
        .reset_index()
    )
    ems_tissue_summary["tissue"] = tissue
    ems_per_tissue.append(ems_tissue_summary)

ems_brain = pd.concat(ems_per_tissue, ignore_index=True)
ems_summary = (
    ems_brain.groupby("var_key")
    .agg(
        max_ems_across_brain=("max_ems_norm", "max"),
        EMS_brain_tissues=("tissue", lambda x: ";".join(sorted(set(x)))),
        EMS_brain_n_tissues=("tissue", lambda x: len(set(x))),
        EMS_brain_genes=("genes", lambda x: ";".join(sorted(set(";".join(x).split(";"))))),
    )
    .reset_index()
)
ems_summary["overlaps_EMS_brain"] = True

mpra_variants = mpra_variants.merge(ems_summary, on="var_key", how="left")
mpra_variants["overlaps_EMS_brain"] = mpra_variants["overlaps_EMS_brain"].fillna(False)
print(mpra_variants["overlaps_EMS_brain"].value_counts())

overlaps_EMS_brain
False    37259
True      1709
Name: count, dtype: int64


In [34]:
# UKBB (94 traits, release 1.1, hg19) -- needs a hg19 liftover of the MPRA variant positions first
mpra_bed = mpra_variants.copy()
mpra_bed["chrom"] = mpra_bed["chr"]
mpra_bed["start"] = mpra_bed["var_key"].apply(lambda x: int(x.split(":")[1]) - 1)
mpra_bed["end"] = mpra_bed["var_key"].apply(lambda x: int(x.split(":")[1]))
mpra_bed["name"] = mpra_bed["var_key"]

mpra_hg38_bed_path = "data/liftover/mpra_hg38.bed.gz"
if writing:
    mpra_bed[["chrom", "start", "end", "name"]].to_csv(mpra_hg38_bed_path, sep="\t", header=False, index=False)
    print(f"Wrote {mpra_hg38_bed_path} -- lift over externally (UCSC liftOver) to hg19 before rerunning")

# result of running UCSC liftOver on the bed file above
mpra_bed_hg19 = pd.read_csv(
    "data/liftover/mpra_hg19_updated.bed.gz", sep="\t", header=None,
    names=["chrom", "hg19_start", "hg19_end", "hg38_name", "number"],
).drop(columns=["number"])
mpra_bed_hg19["var_key"] = mpra_bed_hg19["hg38_name"].apply(lambda x: ":".join(x.split(":")[2:]))
mpra_bed_hg19["hg19_pos"] = mpra_bed_hg19["hg19_start"].astype(int) + 1
# liftOver can emit multiple hg19 candidates per hg38 variant -- keep one per (chrom, hg38 coords)
mpra_bed_hg19["hg38_start"] = mpra_bed_hg19["hg38_name"].apply(lambda x: int(x.split(":")[1].split("-")[0]))
mpra_bed_hg19["hg38_end"] = mpra_bed_hg19["hg38_name"].apply(lambda x: int(x.split(":")[1].split("-")[1]))
mpra_bed_hg19 = mpra_bed_hg19.drop_duplicates(subset=["chrom", "hg38_start", "hg38_end"])

mpra_variants = mpra_variants.merge(
    mpra_bed_hg19[["hg19_start", "hg19_end", "hg19_pos", "var_key"]], on="var_key", how="left"
)
mpra_variants["var_key_hg19"] = mpra_variants.apply(
    lambda r: f"{r['chr']}:{int(r['hg19_pos'])}:{r['ref_allele']}:{r['alt_allele']}" if not pd.isna(r["hg19_pos"]) else None,
    axis=1,
)
mpra_variants["ukbb_overlap_possible"] = mpra_variants["var_key_hg19"].notna()

In [35]:
# NOTE: data too big to store in github downloaded from the finucane lab and stored in data/ https://www.finucanelab.org/data a
ukbb_cols_path = "data/UKBB_94traits_release1.cols"
with open(ukbb_cols_path) as f:
    ukbb_columns = [line.split("\t")[0] for line in f if line.strip()]

ukbb_path = "data/UKBB_94traits_release1.bed.gz"
ukbb_usecols = ["variant", "trait", "pip", "beta_marginal", "beta_posterior", "sd_posterior", "maf", "cs_id", "LD_HWE", "LD_SV"]
mpra_keys_hg19 = set(mpra_variants["var_key_hg19"].dropna())

matched_chunks = []
for chunk in pd.read_csv(
    ukbb_path, sep="\t", compression="gzip", names=ukbb_columns, header=None, usecols=ukbb_usecols, chunksize=2_000_000
):
    matched = chunk.loc[chunk["variant"].isin(mpra_keys_hg19)]
    if not matched.empty:
        matched_chunks.append(matched)
ukbb_filtered = pd.concat(matched_chunks, ignore_index=True) if matched_chunks else pd.DataFrame(columns=ukbb_usecols)
print(f"UKBB-matched variants: {ukbb_filtered['variant'].nunique()}")


def summarize_ukbb_per_variant(df):
    summaries = []
    for variant, sub in df.groupby("variant"):
        max_pip = sub["pip"].max()
        best_rows = sub.loc[sub["pip"] == max_pip]
        any_cs = (sub["cs_id"] != -1).any()
        summaries.append(
            {
                "variant": variant,
                "ukbb_traits": ";".join(sorted(sub["trait"].unique())),
                "ukbb_n_traits": sub["trait"].nunique(),
                "ukbb_best_trait": ";".join(sorted(best_rows["trait"].unique())),
                "ukbb_max_pip": max_pip,
                "ukbb_beta_marginal_at_best_pip": best_rows["beta_marginal"].iloc[0],
                "ukbb_beta_posterior_at_best_pip": best_rows["beta_posterior"].iloc[0],
                "ukbb_sd_posterior_at_best_pip": best_rows["sd_posterior"].iloc[0],
                "ukbb_in_credible_set": any_cs,
                "ukbb_credible_set": ";".join(map(str, sub.loc[sub["cs_id"] != -1, "cs_id"].unique())) if any_cs else pd.NA,
                "ukbb_LD_HWE_flag": sub["LD_HWE"].any(),
                "ukbb_LD_SV_flag": sub["LD_SV"].any(),
            }
        )
    return pd.DataFrame(summaries)


ukbb_summary = summarize_ukbb_per_variant(ukbb_filtered)
ukbb_summary["ukbb_tQTL_overlap"] = True

mpra_variants = mpra_variants.merge(ukbb_summary, left_on="var_key_hg19", right_on="variant", how="left").drop(
    columns=["variant"]
)
for col in ["ukbb_tQTL_overlap", "ukbb_in_credible_set"]:
    mpra_variants[col] = mpra_variants[col].fillna(False)

# any-source eQTL/EMS overlap flag (note: highly imbalanced -- only a small fraction of
# variants have any fine-mapped eQTL signal at all)
mpra_variants["overlapps_eQTL_ems_data"] = (
    mpra_variants["gtex_eQTL_overlap"].fillna(False)
    | mpra_variants["ukbb_tQTL_overlap"].fillna(False)
    | mpra_variants["overlaps_EMS_brain"].fillna(False)
)

FileNotFoundError: [Errno 2] No such file or directory: 'data/UKBB_94traits_release1.cols'

In [36]:
# drop the shared columns between mpra_variants and variant_effect_df from mpra_variants except var_key and merge the tables
variant_effect_df_cols = set(variant_effect_df.columns.to_list())
mpra_variants_cols = set(mpra_variants.columns.to_list())

columns_in_common = list(variant_effect_df_cols.intersection(mpra_variants_cols))

# drop all columns in columns_in_common from mpra_variants
filtered_mpra_variants = mpra_variants.drop(columns=columns_in_common)
# drop duplicates var_key
filtered_mpra_variants = filtered_mpra_variants.drop_duplicates(subset=["var_key"])

variant_effect_df["pos"] = variant_effect_df.apply(get_strand_aware_position, axis=1)
variant_effect_df[["ref_allele", "alt_allele"]] = variant_effect_df.apply(get_strand_aware_alleles, axis=1)

variant_effect_df["var_key"] = (
    variant_effect_df["chr"].astype(str) + ":" + variant_effect_df["pos"].astype(str) + ":"
    + variant_effect_df["ref_allele"] + ":" + variant_effect_df["alt_allele"]
)

variant_effect_combined = variant_effect_df.merge(filtered_mpra_variants, on="var_key", how="left")
variant_effect_combined

,ID,REF,ALT,new_REF,new_ALT,ref_seq,chr,start,end,strand,...,max_ems_across_brain,EMS_brain_tissues,EMS_brain_n_tissues,EMS_brain_genes,overlaps_EMS_brain,hg19_start,hg19_end,hg19_pos,var_key_hg19,ukbb_overlap_possible
0,cardiac_neuro_cava_random:ALT_SKI|ENSG00000157...,cardiac_neuro_cava_random:REF_SKI|ENSG00000157...,cardiac_neuro_cava_random:ALT_SKI|ENSG00000157...,cardiac_neuro_cava_random:REF_SKI|ENSG00000157...,cardiac_neuro_cava_random:ALT_SKI|ENSG00000157...,CCATGCGGTGGCCACAGCCTCGGGTGAGTTCCGGTTCCAAAGTACC...,chr1,2192249,2192519,+,...,NaN,NaN,NaN,NaN,False,2123804.0,2123805.0,2123805.0,chr1:2123805:T:G,True
1,cardiac_neuro_cava_random:ALT_SKI|ENSG00000157...,cardiac_neuro_cava_random:REF_SKI|ENSG00000157...,cardiac_neuro_cava_random:ALT_SKI|ENSG00000157...,cardiac_neuro_cava_random:REF_SKI|ENSG00000157...,cardiac_neuro_cava_random:ALT_SKI|ENSG00000157...,GGACTCCGGTGCCTTCGCATTCCCGAGCTGTTTTTGCTTCTGGAAG...,chr1,2192936,2193206,+,...,NaN,NaN,NaN,NaN,False,2124580.0,2124581.0,2124581.0,chr1:2124581:G:A,True
2,cardiac_neuro_cava_random:ALT_SKI|ENSG00000157...,cardiac_neuro_cava_random:REF_SKI|ENSG00000157...,cardiac_neuro_cava_random:ALT_SKI|ENSG00000157...,cardiac_neuro_cava_random:REF_SKI|ENSG00000157...,cardiac_neuro_cava_random:ALT_SKI|ENSG00000157...,CCTCCACTTGTCAGGAAGCCTGACCCCCAATCCCCTCCCGCCTGAC...,chr1,2197862,2198132,+,...,NaN,NaN,NaN,NaN,False,2129372.0,2129373.0,2129373.0,chr1:2129373:C:A,True
3,cardiac_neuro_cava_random:ALT_SKI|ENSG00000157...,cardiac_neuro_cava_random:REF_SKI|ENSG00000157...,cardiac_neuro_cava_random:ALT_SKI|ENSG00000157...,cardiac_neuro_cava_random:REF_SKI|ENSG00000157...,cardiac_neuro_cava_random:ALT_SKI|ENSG00000157...,CCTCCACTTGTCAGGAAGCCTGACCCCCAATCCCCTCCCGCCTGAC...,chr1,2197862,2198132,+,...,NaN,NaN,NaN,NaN,False,2129445.0,2129446.0,2129446.0,chr1:2129446:C:G,True
4,cardiac_neuro_cava_random:ALT_SKI|ENSG00000157...,cardiac_neuro_cava_random:REF_SKI|ENSG00000157...,cardiac_neuro_cava_random:ALT_SKI|ENSG00000157...,cardiac_neuro_cava_random:REF_SKI|ENSG00000157...,cardiac_neuro_cava_random:ALT_SKI|ENSG00000157...,CCTCCACTTGTCAGGAAGCCTGACCCCCAATCCCCTCCCGCCTGAC...,chr1,2197862,2198132,+,...,NaN,NaN,NaN,NaN,False,2129484.0,2129485.0,2129485.0,chr1:2129485:G:A,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
38963,cardiac_neuro_cava_random:ALT_G6PD|ENSG0000016...,cardiac_neuro_cava_random:REF_G6PD|ENSG0000016...,cardiac_neuro_cava_random:ALT_G6PD|ENSG0000016...,cardiac_neuro_cava_random:REF_G6PD|ENSG0000016...,cardiac_neuro_cava_random:ALT_G6PD|ENSG0000016...,GGAGCTCTGCCTCACCCCACCTGGCCCCAATTGTCCAGCTTGTAGA...,chrX,154545000,154545270,-,...,NaN,NaN,NaN,NaN,False,153773420.0,153773421.0,153773421.0,chrX:153773421:A:G,True
38964,cardiac_neuro_cava_random:ALT_G6PD|ENSG0000016...,cardiac_neuro_cava_random:REF_G6PD|ENSG0000016...,cardiac_neuro_cava_random:ALT_G6PD|ENSG0000016...,cardiac_neuro_cava_random:REF_G6PD|ENSG0000016...,cardiac_neuro_cava_random:ALT_G6PD|ENSG0000016...,ATGTCTGAATTCACCTCCAAATAATGGGAAAACTCCTAGGTATATA...,chrX,154549802,154550072,-,...,NaN,NaN,NaN,NaN,False,153778137.0,153778138.0,153778138.0,chrX:153778138:T:G,True
38965,cardiac_neuro_cava_random:ALT_G6PD|ENSG0000016...,cardiac_neuro_cava_random:REF_G6PD|ENSG0000016...,cardiac_neuro_cava_random:ALT_G6PD|ENSG0000016...,cardiac_neuro_cava_random:REF_G6PD|ENSG0000016...,cardiac_neuro_cava_random:ALT_G6PD|ENSG0000016...,CCTCTGCCCTCCCTGGCTTCTTCCCCTGTCCCTCCTTTCCCTTCCC...,chrX,154552145,154552415,-,...,NaN,NaN,NaN,NaN,False,153780503.0,153780504.0,153780504.0,chrX:153780504:C:T,True
38966,cardiac_neuro_cava_random:ALT_G6PD|ENSG0000016...,cardiac_neuro_cava_random:REF_G6PD|ENSG0000016...,cardiac_neuro_cava_random:ALT_G6PD|ENSG0000016...,cardiac_neuro_cava_random:REF_G6PD|ENSG0000016...,cardiac_neuro_cava_random:ALT_G6PD|ENSG0000016...,CCTCTGCCCTCCCTGGCTTCTTCCCCTGTCCCTCCTTTCCCTTCCC...,chrX,154552145,154552415,-,...,NaN,NaN,NaN,NaN,False,153780585.0,153780586.0,153780586.0,chrX:153780586:G:A,True


## Write the final variant annotation table

In [ ]:
output_dir = "data/"
tested_variant_annotation_output_path = os.path.join(output_dir, "80k_tested_variant_annotation_table_with_readout.tsv.gz")

variant_effect_df_tested_with_readout = variant_effect_combined.loc[
    variant_effect_combined["ID"].str.startswith("cardiac_neuro_cava_random:")
    & variant_effect_combined["NGN2_variant_has_readout"]
].copy()

if writing:
    variant_effect_df_tested_with_readout.to_csv(
        tested_variant_annotation_output_path, sep="\t", index=False, compression="gzip"
    )
    print(
        f"Wrote tested+readout variant annotation table ({variant_effect_df_tested_with_readout.shape[0]} rows) "
        f"to {tested_variant_annotation_output_path}"
    )

print(f"Final table: {variant_effect_combined.shape[0]} rows x {variant_effect_combined.shape[1]} columns")
print(variant_effect_combined.columns.tolist())
print(f"Final table: {variant_effect_df_tested_with_readout.shape[0]} rows x {variant_effect_df_tested_with_readout.shape[1]} columns")
print(variant_effect_df_tested_with_readout.columns.tolist())

Final table: 38968 rows x 137 columns
['ID', 'REF', 'ALT', 'new_REF', 'new_ALT', 'ref_seq', 'chr', 'start', 'end', 'strand', 'alt_seq', 'variant_pos', 'SPDI', 'NGN2_variant_adj_p-value', 'NGN2_variant_logFC', 'NGN2_variant_has_readout', 'NGN2_variant_is_significant', 'NGN2_variant_effect_direction', 'chr_pos_ref_alt', 'DNase_max', 'max_col', 'enformer_classes_list', 'enformer_class', 'NGN2_variant_abs_logFC', 'NGN2_high_effect_var_09', 'NGN2_high_effect_var_095', 'gnomad_SPDI', 'gnomad_AC', 'gnomad_AF', 'gnomad_AF_popmax', 'gnomad_AF_eas', 'gnomad_AF_nfe', 'gnomad_AF_fin', 'gnomad_AF_afr', 'gnomad_AF_asj', 'af_category', 'substitution_type', 'is_transversion', 'H13_motif_id', 'tf_gene_symbol', 'tf_nTPM_brain', 'fimo_start', 'fimo_stop', 'fimo_strand', 'REF_local_bit_score', 'REF_local_relative_bit_score', 'ALT_local_bit_score', 'ALT_local_relative_bit_score', 'REF_full_bit_score', 'REF_full_relative_bit_score', 'ALT_full_bit_score', 'ALT_full_relative_bit_score', 'max_local_bit_score',

## QC: sanity checks on the built table

In [40]:
def sanity_check_variant_pos_column(ref_seq, alt_seq, variant_pos):
    return ref_seq[variant_pos] != alt_seq[variant_pos]


snv_rows = variant_effect_df.loc[has_snv_alleles.reindex(variant_effect_df.index, fill_value=False)]
is_sane = snv_rows.apply(
    lambda row: sanity_check_variant_pos_column(row["ref_seq"], row["alt_seq"], row["variant_pos"]), axis=1
)
assert_condition(is_sane.all(), "variant_pos does not point at a real ref/alt mismatch for some SNVs")
print(f"variant_pos sanity check passed for {is_sane.sum()} SNV rows")

if plotting:
    plt.figure(figsize=(8, 6))
    sns.violinplot(data=variant_effect_df_tested_with_readout, y="NGN2_variant_logFC", inner="quartile")
    plt.title("Distribution of NGN2 variant log2FC (tested variants with readout)")
    plt.ylabel("NGN2 variant log2FC")
    plt.show()

    plt.figure(figsize=(8, 6))
    sns.boxplot(data=variant_effect_df_tested_with_readout, x="af_category", y="abs_delta_local_bit_score", order=["singleton", "rare", "common", "very common"])
    plt.title("Absolute delta local bit score by allele-frequency category")
    plt.show()

variant_pos sanity check passed for 38968 SNV rows
